# STEP 4-5. 결측치 ffill 처리 및 기업 제거

## 전체 흐름

```
기업_통합본_(todo).parquet
        ↓ STEP 5 — ffill 처리
기업_통합본_(todo)_ffill.parquet
        ↓ STEP 4-A — 급료 컬럼 제거
        ↓ STEP 4-B — 기업별 결측치 비율 분석
        ↓ STEP 4-C — NaN 기업 제거 (2010/2011년 외 연도에 NaN 있는 기업)
기업_통합본_(todo)_ffill_정제.parquet
```

## 실행 순서 안내
ffill 처리(STEP 5)가 결측치 분석·제거(STEP 4)의 **입력 데이터**를 생성하므로
이 노트북에서는 **STEP 5 → STEP 4** 순서로 실행합니다.

## 실행 전 경로 확인
이 노트북은 `윤태/` 폴더 안에 있습니다.
모든 입출력 파일은 `윤태/데이터/` 폴더에서 관리됩니다.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('데이터').resolve()
print(f'데이터 경로: {DATA_DIR}')

---
# STEP 5. 결측치 Forward Fill 처리

## 처리 규칙

| 상황 | 처리 |
|------|------|
| 단일 NaN (앞뒤 모두 값 있음) | 직전 연도 값으로 채움 |
| 연속 2개 이상 NaN | NaN 유지 |
| 첫 연도 NaN (직전 없음) | NaN 유지 |

## 핵심 로직
`ffill(limit=1)` 단독으로는 연속 NaN의 첫 번째만 채워 불완전합니다.

**해결책:**
1. `ffill(limit=1)` 으로 단일 NaN 채우기
2. `prev_null | next_null` 조건으로 연속 NaN 위치 식별 → 다시 NaN 복원

**기업 경계 오염 방지:**
`groupby('사업자등록번호').apply()` 로 기업별 그룹 내에서만 shift 연산을 수행합니다.

In [ ]:
print('로드 중...')
df = pd.read_parquet(DATA_DIR / '기업_통합본_(todo).parquet')
df = df.sort_values(['사업자등록번호', '회계년도']).reset_index(drop=True)
print(f'원본: {len(df):,}행  |  기업: {df["사업자등록번호"].nunique():,}개  |  컬럼: {len(df.columns)}개')

# 재무 컬럼 선택
meta_cols = {
    '회사명', '회계년도', '사업자등록번호', '종업원', '설립일',
    '금감원등록번호', '외부감사기관',
    '통계청 한국표준산업분류 코드 11차(대분류)',
    '통계청 한국표준산업분류 코드 11차(중분류)',
    '통계청 한국표준산업분류 11차(중분류)',
}
num_cols = [c for c in df.select_dtypes(include='number').columns if c not in meta_cols]
before_null = df[num_cols].isna().sum().sum()
null_counts_before = df.isna().sum()  # ffill 전후 비교용
print(f'ffill 대상 컬럼: {len(num_cols)}개  |  ffill 전 NaN 합계: {before_null:,}개')

In [ ]:
def selective_ffill_vectorized(group):
    """
    단일 NaN만 ffill, 연속 2개 이상 NaN은 유지.
    groupby.apply 호출 → 기업 경계 오염 없음.
    """
    is_null   = group.isna()
    prev_null = is_null.shift(1).fillna(False)   # 이전 셀도 NaN?
    next_null = is_null.shift(-1).fillna(False)  # 다음 셀도 NaN?
    # 연속 NaN: 현재 NaN이면서 앞 또는 뒤도 NaN인 위치
    consecutive = is_null & (prev_null | next_null)

    filled = group.ffill(limit=1)    # 1개까지 채우기
    filled[consecutive] = np.nan     # 연속 NaN 위치 복원
    return filled

In [ ]:
print('기업별 ffill 처리 중... (수 분 소요될 수 있습니다)')
df[num_cols] = (
    df.groupby('사업자등록번호')[num_cols]
    .apply(selective_ffill_vectorized)
    .reset_index(level=0, drop=True)
    .sort_index()
)

after_null = df[num_cols].isna().sum().sum()
print(f'처리 완료')
print(f'  ffill 전 NaN: {before_null:,}개')
print(f'  ffill 후 NaN: {after_null:,}개')
print(f'  채워진 NaN  : {before_null - after_null:,}개')

In [ ]:
# ffill 후 잔류 결측치 컬럼 목록
total = len(df)
null_remain = df[num_cols].isna().sum()
null_remain = null_remain[null_remain > 0].sort_values(ascending=False)
print(f'ffill 후 NaN 남은 컬럼 ({len(null_remain)}개):')
for col, cnt in null_remain.items():
    print(f'  {cnt:>7,}행 ({cnt/total*100:5.2f}%)  {col}')

### ffill 전후 비교
컬럼별로 채워진 NaN 수를 확인합니다.

In [ ]:
null_counts_after = df.isna().sum()
filled_s = null_counts_before - null_counts_after

result = pd.DataFrame({'ffill 전': null_counts_before, 'ffill 후': null_counts_after, '채워진 수': filled_s})
result = result[result['ffill 전'] > 0].sort_values('채워진 수', ascending=False)

print(f'{"컬럼":<45}  {"ffill 전":>8}  {"ffill 후":>8}  {"채워진 수":>8}')
print('-' * 80)
for col, row in result.iterrows():
    print(f'{col:<45}  {int(row["ffill 전"]):>8,}  {int(row["ffill 후"]):>8,}  {int(row["채워진 수"]):>8,}')
print('-' * 80)
print(f'{"합계":<45}  {int(result["ffill 전"].sum()):>8,}  {int(result["ffill 후"].sum()):>8,}  {int(result["채워진 수"].sum()):>8,}')

---
# STEP 4. 결측치 분석 및 기업 제거

## 4-A. 급료 컬럼 제거
`급료` 컬럼은 IFRS 데이터에만 존재하고 결측치 비율이 매우 높습니다.
재무비율 계산에도 사용하지 않으므로 제거합니다.

## 4-B. 기업별 결측치 비율 분석
**결측치 비율 = NaN 셀 수 / (보유 연도 수 × 재무 컬럼 수) × 100**

30% / 50% / 70% 임계값별 기업 수와 행 수를 확인합니다.

## 4-C. 결측치 기업 제거 (연도 조건부)
| 구분 | 처리 |
|------|------|
| 2010·2011년에만 NaN | **유지** — 초기 미수집 데이터로 허용 |
| 다른 연도에도 NaN | **제거** — 중간 공백은 데이터 품질 문제 |

In [ ]:
print('STEP 4 시작 — STEP 5 결과 인계')
print(f'전체: {len(df):,}행  |  기업: {df["사업자등록번호"].nunique():,}개  |  컬럼: {len(df.columns)}개')

In [ ]:
# 4-A. 급료 컬럼 제거
DROP_COLS = ['급료']
drop_exist = [c for c in DROP_COLS if c in df.columns]
if drop_exist:
    df = df.drop(columns=drop_exist)
    print(f'제거된 컬럼: {drop_exist}  |  남은 컬럼: {len(df.columns)}개')
else:
    print('급료 컬럼 없음 — 이미 제거된 상태')

In [ ]:
# 4-B. 기업별 결측치 비율 분석
meta_cols = {
    '회사명', '회계년도', '사업자등록번호', '종업원', '설립일',
    '금감원등록번호', '외부감사기관',
    '통계청 한국표준산업분류 코드 11차(대분류)',
    '통계청 한국표준산업분류 코드 11차(중분류)',
    '통계청 한국표준산업분류 11차(중분류)',
}
fin_cols = [c for c in df.select_dtypes(include='number').columns if c not in meta_cols]
print(f'재무 컬럼 수: {len(fin_cols)}개')

def company_nan_ratio(group):
    return group[fin_cols].isna().sum().sum() / group[fin_cols].size * 100

print('기업별 결측치 비율 계산 중...')
nan_ratio  = df.groupby('사업자등록번호').apply(company_nan_ratio).rename('결측치비율(%)')
year_count = df.groupby('사업자등록번호')['회계년도'].nunique().rename('회계년도수')
summary    = pd.concat([year_count, nan_ratio], axis=1).reset_index()

total_biz  = len(summary)
total_rows = len(df)

print(f'\n{'구분':<15}  {'기업 수':>8}  {'비율':>6}  {'행 수':>8}  {'행 비율':>6}')
print('-' * 55)
print(f'{'전체':<15}  {total_biz:>8,}  {'100%':>6}  {total_rows:>8,}  {'100%':>6}')
for thresh in [30, 50, 70]:
    biz_ids  = summary.loc[summary['결측치비율(%)'] > thresh, '사업자등록번호']
    cnt_biz  = len(biz_ids)
    cnt_row  = df['사업자등록번호'].isin(biz_ids).sum()
    label    = f'결측치 {thresh}% 초과'
    print(f'{label:<15}  {cnt_biz:>8,}  {cnt_biz/total_biz*100:>5.1f}%  {cnt_row:>8,}  {cnt_row/total_rows*100:>5.1f}%')

In [ ]:
# 결측치 비율 구간별 분포
bins   = [0, 10, 20, 30, 50, 70, 100]
labels = ['0~10%', '10~20%', '20~30%', '30~50%', '50~70%', '70~100%']
summary['구간'] = pd.cut(summary['결측치비율(%)'], bins=bins, labels=labels, right=True)
dist = summary['구간'].value_counts().reindex(labels)
print('결측치 비율 구간별 기업 수:')
for label, cnt in dist.items():
    print(f'  {label:>10} : {cnt:>6,}개  ({cnt/total_biz*100:5.1f}%)')

# 70% 초과 기업 상세
over70_df = summary[summary['결측치비율(%)'] > 70].sort_values('결측치비율(%)', ascending=False)
print(f'\n결측치 70% 초과 기업 ({len(over70_df):,}개):')
print(over70_df[['사업자등록번호', '회계년도수', '결측치비율(%)']].to_string(index=False))

In [ ]:
# 4-C. 결측치 기업 제거 — 연도 조건부
id_col = '사업자등록번호'

# NaN 있는 행에서 기업별 NaN 발생 연도 집합 추출
nan_rows  = df[df[fin_cols].isna().any(axis=1)][[id_col, '회계년도']]
nan_years = nan_rows.groupby(id_col)['회계년도'].apply(set)

EARLY_YEARS = {2010, 2011}
only_early  = nan_years[nan_years.apply(lambda y: y.issubset(EARLY_YEARS))]
to_delete   = nan_years[~nan_years.apply(lambda y: y.issubset(EARLY_YEARS))]

print(f'NaN 있는 기업 {len(nan_years):,}개 중:')
print(f'  2010/2011년에만 NaN → 유지: {len(only_early):,}개')
print(f'  다른 연도에도 NaN    → 제거: {len(to_delete):,}개')
print('\n제거 대상 기업별 NaN 발생 연도:')
for biz, years in to_delete.items():
    print(f'  {biz}: {sorted(years)}')

In [ ]:
before_rows = len(df)
before_biz  = df[id_col].nunique()

df = df[~df[id_col].isin(to_delete.index)].copy()

after_rows = len(df)
after_biz  = df[id_col].nunique()

print('[결측치 기업 제거 결과]')
print(f'  행 수  : {before_rows:,} → {after_rows:,}  (제거: {before_rows - after_rows:,}행)')
print(f'  기업 수: {before_biz:,} → {after_biz:,}  (제거: {before_biz - after_biz:,}개)')
print(f'\n제거 후 재무 컬럼 잔류 NaN: {df[fin_cols].isna().sum().sum():,}셀')

In [ ]:
print(f'최종: {len(df):,}행  |  기업: {df["사업자등록번호"].nunique():,}개')